# DocLens: Kaggle pipeline

Thin driver: all logic is in `backend/`. Settings: **GPU T4 x2**, **Internet on**.

Order: ingest (parse, vlm, chunk, embed, claims) → answers → eval. For the parser/extractor benchmark use `kaggle_arena.ipynb`. Then download `results.zip`, unzip into the repo and commit.

In [ ]:
REPO_URL = "https://github.com/shizen-57/PaperGraph.git"  # renamed repos redirect
!git clone $REPO_URL doclens_repo
%cd doclens_repo/doclens
!pip install -q -r requirements.txt docling vllm

In [ ]:
!python data/download.py
!python -m backend.ingest data/space --stages parse

In [ ]:
import os, subprocess, time, urllib.request

def start_vllm(model, *extra):
    """(Re)start vLLM's OpenAI-compatible server and point backend/llm.py at it."""
    subprocess.run('pkill -f vllm.entrypoints', shell=True)
    time.sleep(10)  # let the GPU memory free up
    log = open('vllm.log', 'w')
    subprocess.Popen(['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', model,
                      '--dtype', 'half', '--max-model-len', '8192', '--port', '8000', *extra],
                     stdout=log, stderr=log)
    for _ in range(90):
        try:
            urllib.request.urlopen('http://localhost:8000/health')
            break
        except Exception:
            time.sleep(10)
    else:
        raise RuntimeError('vLLM did not start: see vllm.log')
    os.environ.update(LLM_BASE_URL='http://localhost:8000/v1', LLM_API_KEY='none', LLM_MODEL=model)

## VLM pass (charts and figures)
Rewrites figure elements in place: if you re-run it, re-run `parse` first.

In [ ]:
start_vllm('Qwen/Qwen2.5-VL-7B-Instruct-AWQ')
!python -m backend.ingest data/space --stages vlm,chunk

In [ ]:
!python -m backend.ingest data/space --stages embed

## Extraction and QA answers (text model)

In [ ]:
start_vllm('Qwen/Qwen2.5-7B-Instruct-AWQ')
!python -m backend.ingest data/space --stages claims
!python -m backend.eval answers

In [ ]:
!python -m backend.eval  # re-run locally after grading labels/answer_grades.csv
!zip -qr /kaggle/working/results.zip results labels/answer_grades.csv

## Optional: live demo tunnel
Exposes this vLLM to your laptop. Set a real `--api-key` first: the URL is public.

In [ ]:
# start_vllm('Qwen/Qwen2.5-7B-Instruct-AWQ', '--api-key', 'CHANGE_ME')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8000 > cf.log 2>&1 &
!sleep 8 && grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' cf.log